# 实验5 扩展：四色识别 → P4 抓取 → P9–P12 高位落料（实机互锁版）

本版本在 **§3 四色识别** 的基础上增加：

- 连续视觉线程；
- 机械臂动作线程；
- 抓取后直接移动到高位 home；
- 直接移动到垃圾桶上空并张爪落料；
- 单任务队列；
- `READY / DETECT / BUSY / RESET / FAULT / DONE` 状态机；
- 多帧稳定性确认；
- 抓取后的 P4 清空确认；
- 防止同一物块重复抓取的互锁；
- 抓取完成后自动回到 `capture_image` 拍摄位；
- 运行日志记录。

## 颜色—目标映射

| 颜色 | 垃圾桶位置 | 使用的 XY 位姿 |
| --- | --- | --- |
| 黄色 | P9 | `object_yellow` |
| 绿色 | P10 | `object_green` |
| 蓝色 | P11 | `object_blue` |
| 红色 | P12 | `object_red` |

## 状态机

| 状态 | 含义 |
| --- | --- |
| `READY` | 空闲，等待 P4 出现物块 |
| `DETECT` | 已发现候选，正在累计连续确认帧 |
| `BUSY` | 机械臂正在抓取/投放，视觉线程不会提交新任务 |
| `RESET` | 机械臂已回拍摄位，等待 P4 连续清空 |
| `FAULT` | 相机或机械臂异常，机械臂卸力，必须人工处理 |
| `DONE` | 达到本轮最大任务数，自动停止接收新目标 |

## 防重复抓取规则

1. 黄色颜色必须连续出现 `YELLOW_CONFIRM_FRAMES` 帧（当前 2 帧），红/绿/蓝连续出现 `CONFIRM_FRAMES` 帧（当前 8 帧）；
2. 稳定后才提交一次任务，并原子切换到 `BUSY`；
3. `BUSY` 期间视觉线程只更新画面，不投票、不入队；
4. 抓放结束并确认回到 `capture_image` 后进入 `RESET`；
5. 必须在 P4 连续看不到物块 `CLEAR_FRAMES` 帧，才回到 `READY`；
6. 如果抓取失败或物块仍留在 P4，系统不会自动重试，避免反复下抓。

> **本文件直接控制真实机械臂，不包含仿真或自动降级。**


---
## 1. 实机依赖与路径


In [ ]:
# ============================ 实机依赖与路径 ============================
import os
import sys
import csv
import json
import math
import time
import queue
import threading
from collections import Counter

import cv2
import numpy as np
from IPython.display import display, Image as IPyImage

# 让本 notebook 同时兼容 Jupyter / VS Code 的执行环境
def _candidate_dirs():
    dirs = [os.path.abspath(os.getcwd())]
    for key in ('__vsc_ipynb_file__', '__file__', '__session__', '__notebook__'):
        value = globals().get(key)
        if isinstance(value, str) and value:
            dirs.append(os.path.dirname(os.path.abspath(value)))
    for item in os.environ.get('ROBOT_CV_DIRS', '').split(os.pathsep):
        if item.strip():
            dirs.append(os.path.abspath(item.strip()))
    unique = []
    seen = set()
    for d in dirs:
        if d and d not in seen and os.path.isdir(d):
            seen.add(d)
            unique.append(d)
    return unique


def _looks_like_course_dir(d):
    required_files = ('camera.py', 'arm5dof_uservo.py', 'arm_application.py', 'config.py')
    return (all(os.path.isfile(os.path.join(d, name)) for name in required_files)
            and os.path.isdir(os.path.join(d, 'config'))
            and os.path.isdir(os.path.join(d, 'vconfig')))


COURSE_DIR = next((d for d in _candidate_dirs() if _looks_like_course_dir(d)), None)
if COURSE_DIR is None:
    raise RuntimeError(
        '找不到实机依赖目录。请把本 notebook 放到实验5所在目录，'
        '或设置环境变量 ROBOT_CV_DIRS 指向包含 camera.py / arm5dof_uservo.py / '
        'arm_application.py / config.py / config/ / vconfig/ 的目录。'
    )
if COURSE_DIR not in sys.path:
    sys.path.insert(0, COURSE_DIR)

CONFIG_DIR = os.path.join(COURSE_DIR, 'config')
VCONFIG_DIR = os.path.join(COURSE_DIR, 'vconfig')
HSV_CONFIG_PATH = os.path.join(COURSE_DIR, 'hsv_config.json')
LOG_PATH = os.path.join(COURSE_DIR, 'p4_color_sort_live_log.csv')

print('实机依赖目录:', COURSE_DIR)
print('机械臂配置  :', CONFIG_DIR)
print('相机配置    :', VCONFIG_DIR)


---
## 2. 实机参数与互锁参数


In [ ]:
# ============================ 实机与互锁参数 ============================
EXECUTE_ON_REAL_HARDWARE = True
CAMERA_ID = 0

# §3 固定识别 ROI，640x480 图像坐标 (x, y, w, h)
ROI = (160, 100, 320, 300)
P4_PIXEL_CENTER = (346.0, 303.0)
P4_MAX_PIXEL_DISTANCE = 110.0

# 色块过滤
MIN_AREA = 800
MAX_AREA = 120000
MIN_WIDTH = 25
MAX_WIDTH = 320
MIN_HEIGHT = 25
MAX_HEIGHT = 320
MIN_FILL = 0.55

# 视觉稳定性与互锁
CONFIRM_FRAMES = 8          # 红/绿/蓝必须连续确认的默认帧数
YELLOW_CONFIRM_FRAMES = 2   # 黄色受现场光照影响，只要求连续确认 2 帧
CLEAR_FRAMES = 10           # 抓取后 P4 必须连续清空的帧数
VISION_PERIOD_S = 0.08      # 视觉循环约 12.5 FPS
STATUS_PERIOD_S = 0.20
MAX_AUTO_TASKS = 8          # 本轮最多自动分拣 8 次；设为 0 表示不限制

# P4 抓取
P4_WORKSPACE_XY = (0.0, 0.0)
P4_PICK_Z = 10.0

# 抓取后高位平移与垃圾桶投放
HOME_POSE_NAME = 'home'
HIGH_HOME_Z = 225.0         # home XY 上的高位版本；比 config.py 的 home Z=200 更高
P4_EXIT_LIFT_DZ = 80.0      # 夹紧后先垂直抬离 P4，减少碰撞风险
DIRECT_MOVE_SECONDS = 1.5   # home 与垃圾桶上空之间直接移动的时间
DROP_SETTLE_S = 1.0         # 张爪后等待物块落入垃圾桶

# 颜色 -> 目标编号 / 垃圾桶所在位置的 XY 位姿
# 注意：只取该位姿的 X/Y 作为垃圾桶上空位置，不使用它的低 Z 投放高度。
DESTINATIONS = {
    'yellow': {'point': 'P9',  'pose': 'object_yellow'},
    'green':  {'point': 'P10', 'pose': 'object_green'},
    'blue':   {'point': 'P11', 'pose': 'object_blue'},
    'red':    {'point': 'P12', 'pose': 'object_red'},
}

COLOR_ORDER = ['red', 'yellow', 'green', 'blue']
COLOR_CN = {'red': '红色', 'yellow': '黄色', 'green': '绿色', 'blue': '蓝色', 'none': '未识别'}
COLOR_BGR = {'red': (0, 0, 255), 'yellow': (0, 255, 255),
             'green': (0, 255, 0), 'blue': (255, 0, 0)}

CONFIRM_FRAMES_BY_COLOR = {c: CONFIRM_FRAMES for c in COLOR_ORDER}
CONFIRM_FRAMES_BY_COLOR['yellow'] = YELLOW_CONFIRM_FRAMES

DEFAULT_HSV = {
    'red':    ((0, 70, 50),    (10, 255, 255)),
    'yellow': ((26, 70, 50),   (34, 255, 255)),
    'green':  ((35, 70, 50),   (77, 255, 255)),
    'blue':   ((100, 70, 50),  (124, 255, 255)),
}

HSV_CONFIG_PATH = os.path.join(COURSE_DIR, 'hsv_config.json')
LOG_PATH = os.path.join(COURSE_DIR, 'p4_color_sort_interlock_log.csv')

if os.path.isfile(HSV_CONFIG_PATH):
    with open(HSV_CONFIG_PATH, 'r', encoding='utf-8') as f:
        _saved = json.load(f)
    HSV_PARAMS = {c: (tuple(_saved[c][0]), tuple(_saved[c][1])) for c in COLOR_ORDER}
    print('已载入 hsv_config.json')
else:
    HSV_PARAMS = {c: (tuple(v[0]), tuple(v[1])) for c, v in DEFAULT_HSV.items()}
    print('未找到 hsv_config.json，使用内置 HSV 阈值')

assert len(COLOR_ORDER) == 4
assert set(DESTINATIONS) == set(COLOR_ORDER)
assert {DESTINATIONS[c]['point'] for c in COLOR_ORDER} == {'P9', 'P10', 'P11', 'P12'}
print('颜色—目标映射检查通过。')


---
## 3. 固定 ROI 四色识别器


In [ ]:
# ============================ §3 四色识别器 ============================
def to_hsv(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)


def build_mask(hsv, color, params):
    lower, upper = params[color]
    lower = np.array(lower, dtype=np.uint8)
    upper = np.array(upper, dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)

    # 红色 H 跨 0 度，需要补 170~180 的第二段
    if color == 'red' and int(lower[0]) <= 10:
        mask = cv2.bitwise_or(
            mask,
            cv2.inRange(hsv,
                        np.array([170, int(lower[1]), int(lower[2])], dtype=np.uint8),
                        np.array([180, int(upper[1]), int(upper[2])], dtype=np.uint8))
        )

    kernel_open = np.ones((5, 5), np.uint8)
    kernel_close = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    return mask


def find_blocks(mask, offset=(0, 0)):
    ox, oy = offset
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    blocks = []
    for contour in contours:
        area = float(cv2.contourArea(contour))
        if area < MIN_AREA or area > MAX_AREA:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        if w < MIN_WIDTH or w > MAX_WIDTH or h < MIN_HEIGHT or h > MAX_HEIGHT:
            continue
        fill = area / float(max(1, w * h))
        if fill < MIN_FILL:
            continue
        blocks.append({
            'rect': (x + ox, y + oy, w, h),
            'center': (x + ox + w // 2, y + oy + h // 2),
            'area': area,
            'fill': round(fill, 3),
        })
    blocks.sort(key=lambda item: item['area'], reverse=True)
    return blocks


class P4ColorRecognizer(object):
    def __init__(self, params, roi=ROI, p4_center=P4_PIXEL_CENTER,
                 max_distance=P4_MAX_PIXEL_DISTANCE):
        self.params = params
        self.roi = roi
        self.p4_center = p4_center
        self.max_distance = float(max_distance)

    def detect(self, img_bgr):
        x, y, w, h = self.roi
        roi_bgr = img_bgr[y:y + h, x:x + w]
        if roi_bgr.size == 0:
            raise RuntimeError('识别 ROI 超出图像范围: %s' % (self.roi,))
        roi_hsv = to_hsv(roi_bgr)
        detections = {}
        for color in COLOR_ORDER:
            blocks = find_blocks(build_mask(roi_hsv, color, self.params), offset=(x, y))
            detections[color] = blocks
        return detections

    def classify(self, img_bgr):
        detections = self.detect(img_bgr)
        candidates = []
        for color, blocks in detections.items():
            for block in blocks:
                dx = block['center'][0] - self.p4_center[0]
                dy = block['center'][1] - self.p4_center[1]
                distance = math.hypot(dx, dy)
                if distance <= self.max_distance:
                    item = dict(block)
                    item['color'] = color
                    item['distance_to_p4'] = round(distance, 1)
                    candidates.append(item)

        if not candidates:
            return 'none', None, detections

        best = max(candidates, key=lambda item: item['area'])
        return best['color'], best, detections

    def annotate(self, img_bgr, detections, label='none', best=None):
        canvas = img_bgr.copy()
        x, y, w, h = self.roi
        cv2.rectangle(canvas, (x, y), (x + w, y + h), (255, 200, 0), 2)
        cv2.putText(canvas, 'P4 ROI', (x + 8, y + 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 0), 2, cv2.LINE_AA)
        cv2.drawMarker(canvas, tuple(int(v) for v in self.p4_center), (255, 255, 255),
                       cv2.MARKER_CROSS, 26, 2)

        for color, blocks in detections.items():
            for block in blocks:
                bx, by, bw, bh = block['rect']
                cv2.rectangle(canvas, (bx, by), (bx + bw, by + bh), COLOR_BGR[color], 2)

        if best is not None:
            bx, by, bw, bh = best['rect']
            cv2.rectangle(canvas, (bx, by), (bx + bw, by + bh), COLOR_BGR[label], 4)
            cv2.putText(canvas,
                        '%s  area=%.0f  d=%.1f' % (label, best['area'], best['distance_to_p4']),
                        (bx, max(28, by - 12)), cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, COLOR_BGR[label], 2, cv2.LINE_AA)

        cv2.putText(canvas, 'Result: %s' % COLOR_CN.get(label, label),
                    (20, 42), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, COLOR_BGR.get(label, (255, 255, 255)), 2, cv2.LINE_AA)
        return canvas


def bgr8_to_jpeg(img_bgr, quality=85):
    ok, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        raise RuntimeError('图像 JPEG 编码失败')
    return bytes(buf)


def show_bgr(img_bgr, width=760):
    bgr = cv2.resize(img_bgr, (640, 480))
    display(IPyImage(data=bgr8_to_jpeg(bgr), format='jpeg', width=width))


recognizer = P4ColorRecognizer(HSV_PARAMS)
print('P4 四色识别器已建立，ROI =', ROI, '，P4 像素中心 =', P4_PIXEL_CENTER)


---
## 4. 连接真实相机与机械臂

会先让机械臂回到 `capture_image` 拍摄位。任何一步失败都会直接报错。


In [ ]:
# ============================ 连接真实硬件 ============================
camera = None
arm = None
arm_app = None


def connect_real_hardware():
    global camera, arm, arm_app

    if camera is not None and arm is not None and arm_app is not None:
        print('实机已经连接。')
        return camera, arm, arm_app

    from camera import Camera
    from arm5dof_uservo import Arm5DoFUServo
    from arm_application import ArmApplication
    import config as robot_config

    required_poses = ['capture_image', HOME_POSE_NAME]
    for item in DESTINATIONS.values():
        required_poses.append(item['pose'])
    missing_poses = [name for name in required_poses if name not in robot_config.ARM_POSE_DICT]
    if missing_poses:
        raise RuntimeError('config.py 缺少位姿: %s' % ', '.join(missing_poses))

    # 1) 真实相机
    try:
        camera = Camera(id=CAMERA_ID, config_folder=VCONFIG_DIR)
        if camera.capture is None or not camera.capture.isOpened():
            raise RuntimeError('摄像头未打开')
        camera.load_cam_calib_data()
        for _ in range(8):
            camera.read()
        print('相机连接成功，CAMERA_ID =', CAMERA_ID)
    except BaseException as exc:
        camera = None
        raise RuntimeError('真实相机连接失败: %s' % repr(exc)) from exc

    # 2) 真实机械臂。SDK 串口失败时可能抛 SystemExit，因此捕获 BaseException。
    try:
        arm = Arm5DoFUServo(config_folder=CONFIG_DIR)
    except BaseException as exc:
        try:
            if camera is not None:
                camera.capture.release()
        finally:
            camera = None
            arm = None
        raise RuntimeError(
            '机械臂连接失败。请检查上电、串口和 config.py 的 DEVICE_PORT_DEFAULT: %s'
            % repr(exc)
        ) from exc

    try:
        arm_app = ArmApplication(arm, config_folder=CONFIG_DIR)
        result = arm.set_tool_pose(pose_name='capture_image', T=2.0)
        if result is False:
            raise RuntimeError('capture_image 位姿不可达')
        time.sleep(0.5)
        print('机械臂连接成功，当前末端位姿 =', np.round(arm.get_tool_pose(), 3))
        print('机械臂已到达 capture_image 拍摄位。')
    except BaseException as exc:
        try:
            arm.set_damping(0)
        except BaseException:
            pass
        try:
            camera.capture.release()
        except BaseException:
            pass
        camera = None
        arm = None
        arm_app = None
        raise RuntimeError('机械臂初始化或回拍摄位失败: %s' % repr(exc)) from exc

    return camera, arm, arm_app


camera, arm, arm_app = connect_real_hardware()


---
## 5. 视觉线程 + 机械臂线程 + 状态机互锁

关键设计：

- 视觉线程只做识别和画面更新，不直接驱动机械臂；
- 稳定目标只提交一次，入队时状态原子切换为 `BUSY`；
- 机械臂线程串行执行 P4 抓取、高位 home、垃圾桶上空落料和回拍摄位，不会并发动作；
- 完成后显式回到 `capture_image`，进入 `RESET`；
- `RESET` 只有在 P4 连续清空后才回到 `READY`；
- 异常进入 `FAULT` 并卸力，不自动重试。


In [ ]:
# ============================ 连续分拣状态机 ============================
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except BaseException:
    widgets = None
    HAS_WIDGETS = False

STATE_READY = 'READY'
STATE_DETECT = 'DETECT'
STATE_BUSY = 'BUSY'
STATE_RESET = 'RESET'
STATE_FAULT = 'FAULT'
STATE_DONE = 'DONE'
STATE_STOPPED = 'STOPPED'

STATE_CN = {
    STATE_READY: '空闲，等待P4物块',
    STATE_DETECT: '连续识别确认中',
    STATE_BUSY: '机械臂动作中',
    STATE_RESET: '等待P4清空',
    STATE_FAULT: '故障，已禁止自动动作',
    STATE_DONE: '本轮任务完成',
    STATE_STOPPED: '已停止',
}


def append_live_log(row):
    fields = ['time', 'task_no', 'color', 'target_point', 'pose_name',
              'p4_xy', 'pick_z', 'result', 'detail']
    write_header = not os.path.isfile(LOG_PATH)
    with open(LOG_PATH, 'a', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


class P4SortingController(object):
    """视觉识别与机械臂动作互锁的连续分拣控制器。"""

    def __init__(self, camera, arm, arm_app, recognizer,
                 confirm_frames=CONFIRM_FRAMES, clear_frames=CLEAR_FRAMES,
                 max_tasks=MAX_AUTO_TASKS, confirm_frames_by_color=None):
        self.camera = camera
        self.arm = arm
        self.arm_app = arm_app
        self.recognizer = recognizer
        self.confirm_frames = int(confirm_frames)
        self.confirm_frames_by_color = {
            str(color): max(1, int(frames))
            for color, frames in (confirm_frames_by_color or {}).items()
        }
        self.clear_frames = int(clear_frames)
        self.max_tasks = int(max_tasks)

        import config as robot_config
        self.robot_config = robot_config
        self.high_home_z = float(HIGH_HOME_Z)
        self.p4_exit_lift_dz = max(20.0, float(P4_EXIT_LIFT_DZ))
        self.direct_move_seconds = max(0.5, float(DIRECT_MOVE_SECONDS))
        self.drop_settle_s = max(0.2, float(DROP_SETTLE_S))

        self._lock = threading.RLock()
        self._stop_event = threading.Event()
        self._task_queue = queue.Queue(maxsize=1)
        self._vision_thread = None
        self._arm_thread = None

        self.state = STATE_STOPPED
        self.state_detail = ''
        self.candidate_color = 'none'
        self.candidate_frames = 0
        self.current_required_frames = self.confirm_frames
        self.clear_count = 0
        self.tasks_done = 0
        self.current_task = None
        self.last_color = 'none'
        self.last_best = None
        self.last_detections = {}
        self.last_annotated = None
        self.transitions = []
        self.task_events = []

        self.image_widget = None
        self.status_widget = None
        self._last_status_time = 0.0

    # ------------------------------------------------------------------
    def _set_state(self, new_state, detail=''):
        with self._lock:
            old_state = self.state
            self.state = new_state
            self.state_detail = detail
            self.transitions.append({
                'time': time.time(), 'from': old_state,
                'to': new_state, 'detail': detail,
            })
            if len(self.transitions) > 300:
                del self.transitions[:-300]
        if old_state != new_state:
            print('[状态] %-7s -> %-7s  %s' % (old_state, new_state, detail))

    def required_frames_for(self, color):
        return int(self.confirm_frames_by_color.get(color, self.confirm_frames))

    def snapshot(self):
        with self._lock:
            return {
                'state': self.state,
                'detail': self.state_detail,
                'candidate_color': self.candidate_color,
                'candidate_frames': self.candidate_frames,
                'required_frames': self.current_required_frames,
                'clear_count': self.clear_count,
                'tasks_done': self.tasks_done,
                'current_task': self.current_task,
                'queue_size': self._task_queue.qsize(),
            }

    # ------------------------------------------------------------------
    def _annotate_with_state(self, frame, color, best, detections):
        canvas = self.recognizer.annotate(frame, detections, label=color, best=best)
        with self._lock:
            state = self.state
            detail = self.state_detail
            frames = self.candidate_frames
            required_frames = self.current_required_frames
            clears = self.clear_count
            tasks = self.tasks_done

        cv2.rectangle(canvas, (0, 0), (639, 70), (35, 35, 35), -1)
        cv2.putText(canvas, 'STATE: %s' % state, (18, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(canvas, detail[:55], (18, 58),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, (230, 230, 230), 1, cv2.LINE_AA)
        cv2.putText(canvas, 'confirm %d/%d  clear %d/%d  tasks %d' %
                    (frames, required_frames, clears, self.clear_frames, tasks),
                    (300, 450), cv2.FONT_HERSHEY_SIMPLEX,
                    0.58, (0, 255, 255), 2, cv2.LINE_AA)
        return canvas

    def _publish_view(self, annotated):
        self.last_annotated = annotated
        if HAS_WIDGETS and self.image_widget is not None:
            try:
                self.image_widget.value = bgr8_to_jpeg(annotated, quality=80)
            except BaseException:
                pass
        now = time.time()
        if HAS_WIDGETS and self.status_widget is not None and now - self._last_status_time >= STATUS_PERIOD_S:
            self._last_status_time = now
            snap = self.snapshot()
            color = self.last_color
            detail = snap['detail'] or STATE_CN.get(snap['state'], '')
            html = (
                '<b>状态：%s</b>　识别：%s　任务：%d　队列：%d<br>'
                '<span style="color:#666">%s</span>'
                % (snap['state'], COLOR_CN.get(color, color), snap['tasks_done'],
                   snap['queue_size'], detail)
            )
            try:
                self.status_widget.value = html
            except BaseException:
                pass

    # ------------------------------------------------------------------
    def _vision_loop(self):
        while not self._stop_event.is_set():
            try:
                ok, frame = self.camera.read()
                if not ok or frame is None:
                    raise RuntimeError('相机读帧失败')
                frame = cv2.resize(frame, (640, 480))
                color, best, detections = self.recognizer.classify(frame)

                with self._lock:
                    self.last_color = color
                    self.last_best = best
                    self.last_detections = detections
                    state = self.state

                    # 只有 READY / DETECT 允许累计新任务。
                    if state in (STATE_READY, STATE_DETECT):
                        if best is None or color == 'none':
                            self.candidate_color = 'none'
                            self.candidate_frames = 0
                            self.current_required_frames = self.confirm_frames
                            if state == STATE_DETECT:
                                self._set_state(STATE_READY, '候选消失，重新等待')
                        else:
                            required_frames = self.required_frames_for(color)
                            self.current_required_frames = required_frames
                            if color == self.candidate_color:
                                self.candidate_frames += 1
                            else:
                                self.candidate_color = color
                                self.candidate_frames = 1

                            if self.candidate_frames < required_frames:
                                if state != STATE_DETECT:
                                    self._set_state(
                                        STATE_DETECT,
                                        '连续确认 %s %d/%d' %
                                        (COLOR_CN[color], self.candidate_frames, required_frames)
                                    )
                            else:
                                # 先切 BUSY，再入队，保证视觉线程不会继续累计或重复提交。
                                self._set_state(
                                    STATE_BUSY,
                                    '稳定目标 %s，等待机械臂线程' % COLOR_CN[color]
                                )
                                event = {
                                    'color': color,
                                    'block': dict(best),
                                    'detected_at': time.time(),
                                }
                                self.candidate_color = 'none'
                                self.candidate_frames = 0
                                try:
                                    self._task_queue.put_nowait(event)
                                except queue.Full:
                                    self._set_state(STATE_FAULT, '任务队列异常：已有未执行任务')
                                    self._stop_event.set()

                    # 抓放结束后，必须看到 P4 连续为空才解锁。
                    elif state == STATE_RESET:
                        if best is None or color == 'none':
                            self.clear_count += 1
                            if self.clear_count >= self.clear_frames:
                                self.clear_count = 0
                                self._set_state(STATE_READY, 'P4已清空，可识别下一个物块')
                        else:
                            self.clear_count = 0
                            self.state_detail = 'P4仍有物块，禁止重新触发'
                    # BUSY / FAULT / DONE / STOPPED 期间不投票、不入队。

                annotated = self._annotate_with_state(frame, color, best, detections)
                self._publish_view(annotated)
            except BaseException as exc:
                try:
                    self.arm.set_damping(0)
                except BaseException:
                    pass
                self._set_state(STATE_FAULT, '视觉线程异常: %s' % repr(exc)[:80])
                self._stop_event.set()
                break

            time.sleep(VISION_PERIOD_S)

    # ------------------------------------------------------------------
    def _ik_solution(self, x, y, z, last_joint_angle=None):
        # 与 Arm5DoFUServo.set_tool_pose 内部处理保持一致：
        # 先做 Z 误差补偿，再自动计算 pitch，最后调用逆运动学。
        tool_posi = self.arm.adjust_tool_posi([float(x), float(y), float(z)])
        pitch = self.arm.auto_gen_pitch(tool_posi)
        if hasattr(self.arm, 'kinematic'):
            solutions = self.arm.kinematic.inverse_kinematic(
                tool_posi[0], tool_posi[1], tool_posi[2],
                pitch, math.pi, last_joint_angle=last_joint_angle
            )
            if solutions:
                return True, pitch, solutions[-1]
            return False, pitch, None
        ok, solution = self.arm.inverse_kinematic(tool_posi, pitch, math.pi)
        return bool(ok), pitch, solution

    def _high_home_xy(self):
        home_pose = self.robot_config.ARM_POSE_DICT[HOME_POSE_NAME]
        return float(home_pose[0]), float(home_pose[1])

    def preflight_transfer_paths(self):
        """启动前只校验高位 home 和四个垃圾桶上空目标，不生成中间路点。"""
        errors = []
        home_x, home_y = self._high_home_xy()
        current_joints = None
        try:
            current_joints = self.arm.get_joint_angle_list()[:5]
        except BaseException:
            pass
        ok, pitch, home_solution = self._ik_solution(
            home_x, home_y, self.high_home_z, current_joints
        )
        if not ok:
            errors.append('high_home')
        print('高 home 校验: (%.1f, %.1f, %.1f) pitch=%.1f° -> %s' %
              (home_x, home_y, self.high_home_z,
               math.degrees(pitch), 'OK' if ok else '不可达'))

        for color in COLOR_ORDER:
            destination = DESTINATIONS[color]
            pose = self.robot_config.ARM_POSE_DICT[destination['pose']]
            target_x, target_y = float(pose[0]), float(pose[1])
            target_ok, target_pitch, _ = self._ik_solution(
                target_x, target_y, self.high_home_z, home_solution
            )
            print('目标上空校验: %s %s 上空=(%.1f, %.1f, %.1f) pitch=%.1f° -> %s' %
                  (destination['point'], COLOR_CN[color], target_x, target_y,
                   self.high_home_z, math.degrees(target_pitch),
                   'OK' if target_ok else '不可达'))
            if not target_ok:
                errors.append('%s(%s)' % (destination['point'], color))

        if errors:
            raise RuntimeError('以下高位目标不可达，已拒绝启动: %s' % ', '.join(errors))
        print('高位 home 与四个垃圾桶上空目标全部通过逆运动学校验。')

    def _move_to_high_home(self):
        home_x, home_y = self._high_home_xy()
        result = self.arm.set_tool_pose(
            [home_x, home_y, self.high_home_z],
            tool_roll=math.pi, T=self.direct_move_seconds
        )
        if result is False:
            raise RuntimeError('移动到高位 home 失败')
        print('已到高位 home: (%.1f, %.1f, Z=%.1f)。' %
              (home_x, home_y, self.high_home_z))
        return home_x, home_y

    def _move_direct_overhead(self, target_x, target_y):
        """不经过中间路点，直接从高位 home 移动到垃圾桶上空。"""
        result = self.arm.set_tool_pose(
            [float(target_x), float(target_y), self.high_home_z],
            tool_roll=math.pi, T=self.direct_move_seconds
        )
        if result is False:
            raise RuntimeError('直接移动到垃圾桶上空失败')
        print('已直接移动到垃圾桶上空: (%.1f, %.1f, Z=%.1f)。' %
              (float(target_x), float(target_y), self.high_home_z))

    def _grab_and_drop_into_bin(self, color):
        """P4 夹取 -> 高位 home -> 垃圾桶上空 -> 张爪 -> 高位 home。"""
        destination = DESTINATIONS[color]
        target_pose = self.robot_config.ARM_POSE_DICT[destination['pose']]
        target_x, target_y = float(target_pose[0]), float(target_pose[1])

        pick_wz = float(P4_PICK_Z) + float(self.arm_app.GRAB_CUBE_DZ)
        lift_wz = pick_wz + self.p4_exit_lift_dz

        # 1) 对准 P4、下降并夹紧
        self.arm_app.gripper_open()
        arm_x, arm_y, _ = self.arm_app.tf_ws2arm(
            P4_WORKSPACE_XY[0], P4_WORKSPACE_XY[1], pick_wz
        )
        theta0 = math.atan2(arm_y, arm_x)
        self.arm.set_joint_angle_soft('joint1', theta0, T=0.8)
        self.arm_app.move2ws(P4_WORKSPACE_XY[0], P4_WORKSPACE_XY[1], lift_wz, t=0.8)
        self.arm_app.move2ws(P4_WORKSPACE_XY[0], P4_WORKSPACE_XY[1], pick_wz, t=0.45)
        self.arm_app.gripper_close()
        time.sleep(0.25)

        # 2) 垂直抬离 P4，然后直接到高位 home
        self.arm_app.move2ws(P4_WORKSPACE_XY[0], P4_WORKSPACE_XY[1], lift_wz, t=0.65)
        self._move_to_high_home()

        # 3) 直接从高位 home 移动到垃圾桶上空
        self._move_direct_overhead(target_x, target_y)

        # 4) 保持夹爪在垃圾桶上空，只张爪落料
        self.arm_app.gripper_open()
        time.sleep(self.drop_settle_s)
        print('%s 上空已张爪落料。' % destination['point'])

        # 5) 张爪状态直接回高位 home，再闭合夹爪
        self._move_to_high_home()
        self.arm_app.gripper_close()

    # ------------------------------------------------------------------
    def _execute_task(self, event):
        color = event['color']
        destination = DESTINATIONS[color]
        task_no = self.tasks_done + 1
        self.current_task = {
            'task_no': task_no, 'color': color,
            'target_point': destination['point'],
        }
        self._set_state(
            STATE_BUSY,
            '第%d次：%s -> %s' % (task_no, COLOR_CN[color], destination['point'])
        )

        row = {
            'time': time.strftime('%Y-%m-%d %H:%M:%S'),
            'task_no': task_no,
            'color': color,
            'target_point': destination['point'],
            'pose_name': destination['pose'],
            'p4_xy': '%.1f,%.1f' % P4_WORKSPACE_XY,
            'pick_z': '%.1f' % P4_PICK_Z,
            'result': 'RUNNING',
            'detail': '',
        }

        try:
            print('\n[执行] 第%d次：P4 %s -> %s (%s)' %
                  (task_no, COLOR_CN[color], destination['point'], destination['pose']))

            # 真实动作：P4 抓取 -> 抬到 home 高度 -> 垃圾桶上空 -> 张爪落料。
            self._grab_and_drop_into_bin(color)

            # 投放结束后显式回拍摄位。
            result = self.arm.set_tool_pose(pose_name='capture_image', T=1.5)
            if result is False:
                raise RuntimeError('capture_image 位姿不可达')

            with self._lock:
                self.tasks_done += 1
                self.current_task = None
                if self.max_tasks > 0 and self.tasks_done >= self.max_tasks:
                    self._set_state(STATE_DONE, '本轮已完成%d次，自动停止接单' % self.tasks_done)
                else:
                    self.clear_count = 0
                    self._set_state(STATE_RESET, '已回拍摄位，等待P4清空')

            row['result'] = 'OK'
            row['detail'] = 'high-home direct overhead drop; capture_image reset confirmed'
            append_live_log(row)
            print('[完成] 第%d次成功，机械臂已回到 capture_image。' % task_no)
        except BaseException as exc:
            row['result'] = 'FAILED'
            row['detail'] = repr(exc)[:200]
            try:
                append_live_log(row)
            except BaseException:
                pass
            try:
                self.arm.set_damping(0)
                print('[异常] 机械臂已卸力。')
            except BaseException:
                pass
            self.current_task = None
            self._set_state(STATE_FAULT, '第%d次动作失败: %s' % (task_no, repr(exc)[:70]))
            self._stop_event.set()

    def _arm_loop(self):
        while not self._stop_event.is_set():
            try:
                event = self._task_queue.get(timeout=0.10)
            except queue.Empty:
                continue
            if event is None:
                break
            if self._stop_event.is_set():
                break
            self._execute_task(event)

    # ------------------------------------------------------------------
    def start(self, initial_state=STATE_READY):
        with self._lock:
            if ((self._vision_thread is not None and self._vision_thread.is_alive()) or
                    (self._arm_thread is not None and self._arm_thread.is_alive())):
                print('控制器已经在运行。')
                return False

            # 启动线程前先做纯逆运动学校验，不移动机械臂。
            try:
                self.preflight_transfer_paths()
            except BaseException as exc:
                self._set_state(STATE_FAULT, '高位目标校验失败: %s' % repr(exc)[:80])
                raise

            self._stop_event.clear()
            self._task_queue = queue.Queue(maxsize=1)
            self.candidate_color = 'none'
            self.candidate_frames = 0
            self.current_required_frames = self.confirm_frames
            self.clear_count = 0
            self.tasks_done = 0
            self.current_task = None
            self.state = initial_state
            if initial_state == STATE_RESET:
                self.state_detail = '重新布防：等待P4清空'
            else:
                self.state_detail = '等待P4物块'

            if HAS_WIDGETS:
                self.image_widget = widgets.Image(format='jpeg', width=720)
                self.status_widget = widgets.HTML()
                display(widgets.VBox([
                    widgets.HBox([self.image_widget, self.status_widget]),
                ]))

            self._arm_thread = threading.Thread(target=self._arm_loop, daemon=True)
            self._vision_thread = threading.Thread(target=self._vision_loop, daemon=True)
            self._arm_thread.start()
            self._vision_thread.start()

        print('连续分拣已启动：READY，等待 P4 物块。')
        print('投放方式：抓取 -> 高位 home -> 直接到垃圾桶上空 -> 张爪 -> 高位 home -> 拍摄位。')
        print('确认帧：黄色 %d 帧，红/绿/蓝 %d 帧。' % (YELLOW_CONFIRM_FRAMES, CONFIRM_FRAMES))
        print('摄像头持续运行；BUSY 期间不会提交新任务。')
        return True

    def stop(self, wait=True):
        self._stop_event.set()
        try:
            self._task_queue.put_nowait(None)
        except queue.Full:
            pass
        if wait:
            for thread in (self._vision_thread, self._arm_thread):
                if thread is not None and thread.is_alive():
                    thread.join(timeout=12.0)
        if not self._stop_event.is_set():
            self._stop_event.set()
        with self._lock:
            if self.state not in (STATE_FAULT, STATE_DONE):
                self.state = STATE_STOPPED
                self.state_detail = '人工停止'
        print('连续分拣已停止。')

    def emergency_stop(self):
        self._stop_event.set()
        try:
            self.arm.set_damping(0)
            print('急停：机械臂已卸力。')
        except BaseException as exc:
            print('急停卸力失败:', repr(exc))
        self._set_state(STATE_FAULT, '人工急停')

    def manual_reset(self):
        """故障处理/本轮完成后重新布防：先进入 RESET，P4 清空后自动回 READY。"""
        with self._lock:
            self.candidate_color = 'none'
            self.candidate_frames = 0
            self.clear_count = 0
            self.current_task = None
            self.state = STATE_RESET
            self.state_detail = '人工重新布防，等待P4清空'
        if not self._stop_event.is_set() and self._vision_thread is not None and self._vision_thread.is_alive():
            print('已重新布防：P4连续清空后回到 READY。')
        else:
            print('已重新布防，正在重启视觉与动作线程。')
            self.start(initial_state=STATE_RESET)


print('P4SortingController 已就绪。')


---
## 6. 启动连续分拣

运行本单元后，摄像头会持续工作：

1. 物块放到 P4；
2. 按颜色连续确认：黄色 2 帧，红/绿/蓝 8 帧；
3. 状态切到 `BUSY`，机械臂从 P4 抓取并夹紧物块；
4. 抬升到 home 位，在 home 高度按固定高度路径横移到垃圾桶上空，张爪落料；
5. 回到高位 home，再回到拍摄位，进入 `RESET`；
6. P4 连续清空 10 帧后回到 `READY`，等待下一块。

本轮默认最多自动完成 8 次，防止无人值守时无限触发。


In [ ]:
# ============================ 启动连续分拣 ============================
if not EXECUTE_ON_REAL_HARDWARE:
    raise RuntimeError('EXECUTE_ON_REAL_HARDWARE 为 False')

if 'controller' in globals() and controller is not None:
    controller.stop()

controller = P4SortingController(
    camera=camera,
    arm=arm,
    arm_app=arm_app,
    recognizer=recognizer,
    confirm_frames=CONFIRM_FRAMES,
    confirm_frames_by_color=CONFIRM_FRAMES_BY_COLOR,
    clear_frames=CLEAR_FRAMES,
    max_tasks=MAX_AUTO_TASKS,
)
controller.start()


---
## 7. 状态查看、人工停止与重新布防

- `controller.snapshot()`：查看当前状态和计数；
- `controller.stop()`：停止本轮连续分拣；
- `controller.emergency_stop()`：立即让机械臂卸力；
- `controller.manual_reset()`：排除故障、清空 P4 后重新布防。

出现 `FAULT` 时，系统不会自动重试同一物块。


In [ ]:
# 运行需要时取消对应注释。
# print(controller.snapshot())
# controller.stop()
# controller.emergency_stop()
# controller.manual_reset()


---
## 8. 结束实验

停止视觉线程、释放相机并让机械臂卸力。


In [ ]:
def shutdown_real_hardware():
    global camera, arm, arm_app, controller
    if 'controller' in globals() and controller is not None:
        controller.stop()
        controller = None
    if camera is not None:
        try:
            camera.capture.release()
            print('相机已释放。')
        except BaseException as exc:
            print('相机释放失败:', repr(exc))
        camera = None
    if arm is not None:
        try:
            arm.set_damping(0)
            print('机械臂已卸力。')
        except BaseException as exc:
            print('机械臂卸力失败:', repr(exc))
        arm = None
    arm_app = None


# 全部实验完成后取消下一行注释并运行。
# shutdown_real_hardware()
